<a href="https://colab.research.google.com/github/ProfAndersonVanin/IBM3130-PLN-2026/blob/main/semana-06/Aula06_Embendings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Word2Vec com Gensim**

In [ ]:
# Instalar no Colab
!pip install gensim -q

from gensim.models import Word2Vec

In [ ]:
# Corpus simples — lista de listas de tokens
corpus = [
    ["o", "gato", "dorme", "no", "sofá"],
    ["o", "cachorro", "corre", "no", "parque"],
    ["o", "gato", "come", "o", "rato"],
    ["o", "cachorro", "late", "muito"],
    ["o", "gato", "e", "o", "cachorro", "brincam"],
    ["o", "rato", "corre", "do", "gato"],
    ["o", "cachorro", "dorme", "no", "chão"],
    ["o", "gato", "late", "igual", "ao", "cachorro"],
]

In [ ]:
# Treinar o modelo Word2Vec
modelo = Word2Vec(
    sentences = corpus,    # corpus — lista de listas de tokens
    vector_size = 10,      # dimensões do embedding (baixo para exemplo didático)
    window = 2,            # janela de contexto — palavras de cada lado
    min_count = 1,         # mínimo de ocorrências para incluir a palavra
    sg = 1,                # 1 = Skip-gram, 0 = CBOW
    epochs = 100,          # número de passagens pelo corpus
    seed = 42              # para reprodutibilidade
)

In [ ]:
print("Modelo treinado!")
print("Vocabulário:", list(modelo.wv.key_to_index.keys()))
print("Dimensões do embedding:", modelo.wv.vector_size)

# **Acessando os vetores**

In [ ]:
# Ver o vetor de uma palavra
vetor_gato = modelo.wv["gato"]
print("Vetor de 'gato':")
print(vetor_gato)
print("Formato:", vetor_gato.shape)

In [ ]:
# Ver o vetor de outra palavra
vetor_cachorro = modelo.wv["cachorro"]
print("\nVetor de 'cachorro':")
print(vetor_cachorro)

# **Palavras mais similares**

In [ ]:
# Encontrar palavras mais parecidas com 'gato'
similares_gato = modelo.wv.most_similar("gato", topn=3)

In [ ]:
similares_gato

In [ ]:
print("Palavras mais similares a 'gato':")
for palavra, similaridade in similares_gato:
    print(f"  {palavra:<15} similaridade: {similaridade:.4f}")

In [ ]:
# Fazer o mesmo para 'cachorro'
similares_cachorro = modelo.wv.most_similar("cachorro", topn=3)

In [ ]:
print("\nPalavras mais similares a 'cachorro':")
for palavra, similaridade in similares_cachorro:
    print(f"  {palavra:<15} similaridade: {similaridade:.4f}")

In [ ]:
print(vetor_gato)

In [ ]:
print(vetor_cachorro)

# **Exemplo numérico completo**

In [ ]:
import math

In [ ]:
# Vetores simplificados de duas palavras (apenas 3 dimensões para facilitar)
vetor_gato     = [0.8, 0.3, 0.5]
vetor_cachorro = [0.7, 0.4, 0.6]
vetor_carro    = [0.1, 0.9, 0.2]

In [ ]:
def similaridade_cosseno(v1, v2):

    # Passo 1: produto escalar (A · B)
    produto = 0
    for i in range(len(v1)):
        produto = produto + v1[i] * v2[i]

    # Passo 2: norma do vetor v1 (|A|)
    soma_quadrados_v1 = 0
    for x in v1:
        soma_quadrados_v1 = soma_quadrados_v1 + x * x
    norma_v1 = math.sqrt(soma_quadrados_v1)

    # Passo 3: norma do vetor v2 (|B|)
    soma_quadrados_v2 = 0
    for x in v2:
        soma_quadrados_v2 = soma_quadrados_v2 + x * x
    norma_v2 = math.sqrt(soma_quadrados_v2)

    # Passo 4: dividir
    return produto / (norma_v1 * norma_v2)

In [ ]:
# Calcular similaridades
sim_gato_cachorro = similaridade_cosseno(vetor_gato, vetor_cachorro)
sim_gato_carro    = similaridade_cosseno(vetor_gato, vetor_carro)

print("=== Similaridade de Cosseno ===")
print()
print(f"  'gato' vs 'cachorro' = {sim_gato_cachorro:.4f}")
print(f"  'gato' vs 'carro'    = {sim_gato_carro:.4f}")
print()

In [ ]:
if sim_gato_cachorro > sim_gato_carro:
    print("  → 'cachorro' é mais similar a 'gato' do que 'carro'")
    print("    (faz sentido — ambos são animais de estimação!)")

# **Similaridade com Gensim**

In [ ]:
# Similaridade entre duas palavras usando Gensim
sim = modelo.wv.similarity("gato", "cachorro")
print(f"Similaridade('gato', 'cachorro') = {sim:.4f}")

In [ ]:
sim2 = modelo.wv.similarity("gato", "rato")
print(f"Similaridade('gato', 'rato')     = {sim2:.4f}")

In [ ]:
sim3 = modelo.wv.similarity("gato", "sofá")
print(f"Similaridade('gato', 'sofá')     = {sim3:.4f}")

In [ ]:
# Qual par é mais similar?
pares = [
    ("gato", "cachorro"),
    ("gato", "rato"),
    ("gato", "sofá"),
    ("cachorro", "rato"),
]

In [ ]:
print("\nRanking de similaridade:")
resultados = []
for p1, p2 in pares:
    sim = modelo.wv.similarity(p1, p2)
    resultados.append((p1, p2, sim))

resultados.sort(key=lambda x: x[2], reverse=True)

posicao = 1
for p1, p2, sim in resultados:
    print(f"  {posicao}. '{p1}' vs '{p2}': {sim:.4f}")
    posicao = posicao + 1

> **Similaridade de cosseno varia entre -1 e +1**



```
+1.0  →  vetores apontam na mesma direção  →  muito similares
 0.0  →  vetores perpendiculares           →  sem relação
-1.0  →  vetores apontam em direções opostas →  significados opostos
```



Isso não significa que 'gato' e 'cachorro' são opostos — significa que o modelo não aprendeu nenhuma relação entre eles. Os vetores estão essencialmente apontando em direções aleatórias no espaço vetorial.

# **Analogias Vetoriais**

# **Embeddings Pré-treinados**

## Usando embeddings pré-treinados com Gensim

In [ ]:
import gensim.downloader as api

# Baixar embeddings pré-treinados (pode demorar — arquivo grande)
# 'glove-wiki-gigaword-50' é o menor disponível: 50 dimensões
modelo_pt = api.load("glove-wiki-gigaword-50")

In [ ]:
print("Modelo carregado!")
print("Vocabulário:", len(modelo_pt.key_to_index), "palavras")
print("Dimensões:", modelo_pt.vector_size)

In [ ]:
# Ver as primeiras 20 palavras
print("\nPrimeiras 20 palavras do vocabulário:")
palavras = list(modelo_pt.key_to_index.keys())
for i in range(20):
    print(f"  [{i:>2}] {palavras[i]}")

In [ ]:
# Verificar se uma palavra existe
for palavra in ["king", "queen", "man", "woman", "cat", "dog"]:
    existe = palavra in modelo_pt
    status = "✓" if existe else "✗"
    print(f"  {status} '{palavra}'")

In [ ]:
# Testar similaridade
print("\nSimilaridade 'king' vs 'queen':", modelo_pt.similarity("king", "queen"))
print("Similaridade 'cat' vs 'dog':   ", modelo_pt.similarity("cat",  "dog"))
print("Similaridade 'cat' vs 'car':   ", modelo_pt.similarity("cat",  "car"))

In [ ]:
# Analogia: king - man + woman = ?
resultado = modelo_pt.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=3
)
print("\nking - man + woman ≈")
for palavra, sim in resultado:
    print(f"  {palavra} ({sim:.4f})")

## Embeddings para português — NILC/USP

Os embeddings do NILC agora estão disponíveis no Hugging Face, que é muito mais estável que o servidor da USP. Use assim:

In [ ]:
# Instalar biblioteca do Hugging Face
!pip install huggingface_hub safetensors -q

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file
from gensim.models import KeyedVectors
import numpy as np

In [ ]:
# Baixar embeddings e vocabulário
# repo_id correto: nilc-nlp/word2vec-skip-gram-50d  (com 'd' no final)
path_emb   = hf_hub_download(repo_id="nilc-nlp/word2vec-skip-gram-50d", filename="embeddings.safetensors")
path_vocab = hf_hub_download(repo_id="nilc-nlp/word2vec-skip-gram-50d", filename="vocab.txt")

In [ ]:
# Carregar vetores e vocabulário
data    = load_file(path_emb)
vetores = data["embeddings"]

In [ ]:
with open(path_vocab, encoding="utf-8") as f:
    vocab = [linha.strip() for linha in f]

print(f"Vocabulário: {len(vocab)} palavras")
print(f"Dimensões:   {vetores.shape}")

In [ ]:
# Criar o objeto KeyedVectors com os dados carregados
modelo_pt = KeyedVectors(vector_size=vetores.shape[1])
modelo_pt.add_vectors(vocab, vetores)

print("Modelo montado!")
print(f"Total de palavras: {len(modelo_pt.key_to_index)}")
print(f"Dimensões:         {modelo_pt.vector_size}")

**Verificar palavras no vocabulário**

In [ ]:
# Confirmar que as palavras que vamos usar existem no modelo
palavras_teste = ["rei", "rainha", "homem", "mulher",
                  "gato", "cachorro", "bom", "excelente"]

print("Palavras no vocabulário:")
for palavra in palavras_teste:
    status = "✓" if palavra in modelo_pt else "✗"
    print(f"  {status} '{palavra}'")

In [ ]:
print("Palavras similares a 'inteligência' em português:")
for palavra, sim in modelo_pt.most_similar("inteligência", topn=5):
    print(f"  {palavra:<20} {sim:.4f}")

In [ ]:
# Comparar pares de palavras
pares = [
    ("gato",  "cachorro"),
    ("rei",   "rainha"),
    ("bom",   "excelente"),
    ("bom",   "péssimo"),
    ("gato",  "computador"),
]

print("Similaridade de cosseno entre pares:")
print()
for p1, p2 in pares:
    sim = modelo_pt.similarity(p1, p2)
    print(f"  '{p1}' vs '{p2}': {sim:.4f}")

## Analogias vetoriais em português

In [ ]:
# Agora sim — analogias funcionando em português!
analogias = [
    {
        "positive": ["rainha", "homem"],
        "negative": ["rei"],
        "descricao": "rainha - rei + homem ≈ ?"
    },
    {
        "positive": ["paris", "brasil"],
        "negative": ["frança"],
        "descricao": "paris - frança + brasil ≈ ?"
    },
    {
        "positive": ["atriz", "homem"],
        "negative": ["mulher"],
        "descricao": "atriz - mulher + homem ≈ ?"
    },
]

In [ ]:
for a in analogias:
    print(a["descricao"])
    resultado = modelo_pt.most_similar(
        positive = a["positive"],
        negative = a["negative"],
        topn     = 3
    )
    for palavra, sim in resultado:
        print(f"  {palavra:<20} {sim:.4f}")
    print()

In [ ]:
analogias = [

    # ── GÊNERO ───────────────────────────────────────────────
    {
        "positive": ["médica", "homem"],
        "negative": ["mulher"],
        "descricao": "médica - mulher + homem ≈ ?  (esperado: médico)"
    },
    {
        "positive": ["professor", "mulher"],
        "negative": ["homem"],
        "descricao": "professor - homem + mulher ≈ ?  (esperado: professora)"
    },
    {
        "positive": ["presidente", "mulher"],
        "negative": ["homem"],
        "descricao": "presidente - homem + mulher ≈ ?  (esperado: presidenta)"
    },

    # ── PAÍSES E CAPITAIS ────────────────────────────────────
    {
        "positive": ["brasília", "argentina"],
        "negative": ["brasil"],
        "descricao": "brasília - brasil + argentina ≈ ?  (esperado: buenos aires)"
    },
    {
        "positive": ["brasília", "alemanha"],
        "negative": ["brasil"],
        "descricao": "brasília - brasil + alemanha ≈ ?  (esperado: berlim)"
    },
    {
        "positive": ["brasília", "japão"],
        "negative": ["brasil"],
        "descricao": "brasília - brasil + japão ≈ ?  (esperado: tóquio)"
    },

    # ── FORMAS VERBAIS ───────────────────────────────────────
    {
        "positive": ["correu", "andar"],
        "negative": ["correr"],
        "descricao": "correu - correr + andar ≈ ?  (esperado: andou)"
    },
    {
        "positive": ["comeu", "beber"],
        "negative": ["comer"],
        "descricao": "comeu - comer + beber ≈ ?  (esperado: bebeu)"
    },

    # ── FAMÍLIA ──────────────────────────────────────────────
    {
        "positive": ["mãe", "homem"],
        "negative": ["mulher"],
        "descricao": "mãe - mulher + homem ≈ ?  (esperado: pai)"
    },
    {
        "positive": ["filha", "homem"],
        "negative": ["mulher"],
        "descricao": "filha - mulher + homem ≈ ?  (esperado: filho)"
    },
    {
        "positive": ["avó", "homem"],
        "negative": ["mulher"],
        "descricao": "avó - mulher + homem ≈ ?  (esperado: avô)"
    },

    # ── GRAU DOS ADJETIVOS ───────────────────────────────────
    {
        "positive": ["melhor", "ruim"],
        "negative": ["bom"],
        "descricao": "melhor - bom + ruim ≈ ?  (esperado: pior)"
    },
    {
        "positive": ["maior", "pequeno"],
        "negative": ["grande"],
        "descricao": "maior - grande + pequeno ≈ ?  (esperado: menor)"
    },

    # ── ÁREA PROFISSIONAL ────────────────────────────────────
    {
        "positive": ["hospital", "professor"],
        "negative": ["médico"],
        "descricao": "hospital - médico + professor ≈ ?  (esperado: escola)"
    },
    {
        "positive": ["advogado", "tribunal"],
        "negative": ["médico"],
        "descricao": "advogado - médico + tribunal ≈ ?  (esperado: hospital ou consultório)"
    },

    # ── TECNOLOGIA ───────────────────────────────────────────
    {
        "positive": ["computador", "ligar"],
        "negative": ["desligar"],
        "descricao": "computador - desligar + ligar ≈ ?  (esperado: inicializar, boot)"
    },
    {
        "positive": ["software", "construção"],
        "negative": ["arquitetura"],
        "descricao": "software - arquitetura + construção ≈ ?  (esperado: desenvolvimento)"
    },

    # ── SINGULAR E PLURAL ────────────────────────────────────
    {
        "positive": ["cidades", "país"],
        "negative": ["cidade"],
        "descricao": "cidades - cidade + país ≈ ?  (esperado: países)"
    },
    {
        "positive": ["alunos", "professor"],
        "negative": ["aluno"],
        "descricao": "alunos - aluno + professor ≈ ?  (esperado: professores)"
    },
]

# Rodar todas as analogias
for a in analogias:
    print(a["descricao"])
    try:
        resultado = modelo_pt.most_similar(
            positive = a["positive"],
            negative = a["negative"],
            topn     = 3
        )
        for palavra, sim in resultado:
            print(f"  {palavra:<20} {sim:.4f}")
    except KeyError as e:
        print(f"  ✗ Palavra não encontrada no vocabulário: {e}")
    print()

## Usando GloVe com Gensim

Baixar o arquivo

```
glove.6b.50d.txt
```
de nlp.stanford.edu/projects/glove/

O arquivo está disponível em https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip

In [ ]:
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors

## carregar o glove.6B.50d.txt

In [ ]:
# Converter formato GloVe para formato Word2Vec
glove2word2vec("glove.6B.50d.txt", "glove_w2v.txt")


In [ ]:
# Carregar com Gensim
glove = KeyedVectors.load_word2vec_format("glove_w2v.txt", binary=False)

print(f"Modelo carregado: {len(glove.key_to_index)} palavras · {glove.vector_size} dimensões")

In [ ]:
# Usar normalmente
print(glove.most_similar("computer", topn=5))


In [ ]:
print(glove.similarity("good", "excellent"))

# **Visualizando Embeddings com PCA**

## Visualizando Word2Vec com PCA

In [ ]:
import math
from gensim.models import Word2Vec

In [ ]:
# Corpus sobre tecnologia e animais — para ver grupos distintos
corpus = [
    ["inteligência", "artificial", "aprende", "dados"],
    ["rede", "neural", "processa", "texto"],
    ["algoritmo", "machine", "learning", "treina"],
    ["modelo", "linguagem", "texto", "palavras"],
    ["computador", "processa", "dados", "rápido"],
    ["gato", "dorme", "sofá", "ronrona"],
    ["cachorro", "late", "corre", "brinca"],
    ["pássaro", "voa", "canta", "ninho"],
    ["peixe", "nada", "água", "aquário"],
    ["coelho", "pula", "come", "cenoura"],
]

In [ ]:
# Treinar Word2Vec
modelo = Word2Vec(
    sentences    = corpus,
    vector_size  = 20,
    window       = 2,
    min_count    = 1,
    sg           = 1,
    epochs       = 200,
    seed         = 42
)

In [ ]:
# Palavras que vamos visualizar
palavras_tecnologia = ["inteligência", "rede", "algoritmo", "modelo", "computador"]
palavras_animais    = ["gato", "cachorro", "pássaro", "peixe", "coelho"]
palavras_todas      = palavras_tecnologia + palavras_animais

In [ ]:
# Extrair vetores
vetores = []
for palavra in palavras_todas:
    if palavra in modelo.wv:
        vetores.append(modelo.wv[palavra].tolist())

print(f"Total de vetores: {len(vetores)}")
print(f"Dimensões originais: {len(vetores[0])}")

In [ ]:
vetores

In [ ]:
# Observar os vetores individuais de cada palavra
print('=== Vetores individuais ===')
print()

for i in range(len(palavras_todas)):
    palavra = palavras_todas[i]
    vetor   = vetores[i]

    # Identificar o grupo
    if palavra in palavras_tecnologia:
        grupo = 'Tecnologia'
    else:
        grupo = 'Animal'

    print(f'Palavra: "{palavra}"  [{grupo}]')
    print(f'  Dimensões: {len(vetor)}')

    # Mostrar os valores arredondados
    valores_arredondados = []
    for v in vetor:
        valores_arredondados.append(round(v, 4))
    print(f'  Vetor: {valores_arredondados}')
    print()

In [ ]:
import matplotlib.pyplot as plt

# Usando sklearn apenas para o PCA (não há forma simples sem ele)
from sklearn.decomposition import PCA

import numpy as np

In [ ]:
# Converter para array numpy
X = np.array(vetores)

In [ ]:
X

In [ ]:
# Aplicar PCA — reduzir para 2 dimensões
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

In [ ]:
X_2d

In [ ]:
# Plotar
fig, ax = plt.subplots(figsize=(10, 7))

# Plotar tecnologia em azul
for i in range(len(palavras_tecnologia)):
    ax.scatter(X_2d[i, 0], X_2d[i, 1], color='#2E75B6', s=100)
    ax.annotate(palavras_todas[i],
                xy=(X_2d[i, 0], X_2d[i, 1]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=11, color='#2E75B6')

In [ ]:
# Plotar
fig, ax = plt.subplots(figsize=(10, 7))
# Plotar animais em verde
for i in range(len(palavras_tecnologia), len(palavras_todas)):
    ax.scatter(X_2d[i, 0], X_2d[i, 1], color='#1A5E3A', s=100)
    ax.annotate(palavras_todas[i],
                xy=(X_2d[i, 0], X_2d[i, 1]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=11, color='#1A5E3A')

In [ ]:
# Plotar
fig, ax = plt.subplots(figsize=(10, 7))

# Plotar tecnologia em azul
for i in range(len(palavras_tecnologia)):
    ax.scatter(X_2d[i, 0], X_2d[i, 1], color='#2E75B6', s=100)
    ax.annotate(palavras_todas[i],
                xy=(X_2d[i, 0], X_2d[i, 1]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=11, color='#2E75B6')

# Plotar animais em verde
for i in range(len(palavras_tecnologia), len(palavras_todas)):
    ax.scatter(X_2d[i, 0], X_2d[i, 1], color='#1A5E3A', s=100)
    ax.annotate(palavras_todas[i],
                xy=(X_2d[i, 0], X_2d[i, 1]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=11, color='#1A5E3A')

ax.set_title('Word Embeddings visualizados em 2D (PCA)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Componente Principal 1')
ax.set_ylabel('Componente Principal 2')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.grid(True, alpha=0.3)

# Legenda manual
from matplotlib.patches import Patch
legenda = [
    Patch(color='#2E75B6', label='Tecnologia / IA'),
    Patch(color='#1A5E3A', label='Animais'),
]
ax.legend(handles=legenda, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
print("Observe como palavras do mesmo campo semântico")
print("ficam agrupadas no gráfico!")

# **Aplicações Práticas**

In [ ]:
# Aplicação 1: busca semântica
# Encontrar documentos semanticamente similares à query

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt',     quiet=True)

print('Recursos baixados!')

query    = "inteligência artificial aprende"
docs     = ["machine learning processa dados",
            "gato dorme no sofá",
            "rede neural treina modelo"]

# Vetorizar query e documentos com word2vec
# (média dos vetores das palavras)
def media_vetores(tokens, modelo):
    vetores_validos = []
    for token in tokens:
        if token in modelo.wv:
            vetores_validos.append(modelo.wv[token])
    if not vetores_validos:
        return None
    # Calcular média manualmente
    n    = len(vetores_validos)
    dims = len(vetores_validos[0])
    media = []
    for d in range(dims):
        soma = 0
        for v in vetores_validos:
            soma = soma + v[d]
        media.append(soma / n)
    return media

from nltk.tokenize import word_tokenize

tokens_query = word_tokenize(query.lower(), language='portuguese')
vetor_query  = media_vetores(tokens_query, modelo)

print("Busca semântica para:", query)
print()

resultados = []
for doc in docs:
    tokens_doc  = word_tokenize(doc.lower(), language='portuguese')
    vetor_doc   = media_vetores(tokens_doc, modelo)
    if vetor_doc and vetor_query:
        sim = similaridade_cosseno(vetor_query, vetor_doc)
        resultados.append((doc, sim))

resultados.sort(key=lambda x: x[1], reverse=True)

for doc, sim in resultados:
    print(f"  {sim:.4f}  '{doc}'")